# **Measure Organelle Interactions**

***Prior to this notebook, you should have already run through [2.0_quantification_setup](2.0_quantification_setup.ipynb).***

In notebooks 2.1 through 2.4, we will go over the implementation of `infer-subc` quantification methods (explained in detail in the `method_...` notebooks) to assess the morphology, interactions, and distribution of organelles at the single-cell level. 

### 📍 **Purpose**
This notebook can be used to measure the `interactions` -- the overlapping of two or more segmented `organelles` -- from one or more cells. It includes options to:
1. 🦠 Quantify the interactions of *two or more organelle(s)* from <ins>ONE CELL</ins>
2. 🧪 Batch process the interactions of *two or more organelle(s)* from *multiple cells* for a <ins>SINGLE EXPERIMENT</ins>
3. 🧮 Summarize interaction metrics *per cell* across <INS>ONE OR MORE EXPERIMENTS</ins>


### 🍃 **Biological Relevance - Organelle Interactions**
Intracellular organelles do not exist independently of one another. In recent years, the existance and function of organelle contact site, regions of close apposition between membrane bound organelles has been recognized. They facilitate protein, lipid, and metabolite transport, coordinate organelle trafficking and function, and are involved in many cellular pathways and functions [[1](https://www.cell.com/cell/pdf/S0092-8674(23)01328-4.pdf)]. 

The distance between membranes at contact sites is between 30-80 nm depending on the types of organelles involved. This distance is not resolvable using standard confocal microscopy images [[2](https://zeiss-campus.magnet.fsu.edu/articles/basics/resolution.html)]. However, interactions between organelle which can include organelle contact sites can be estimated through overlap in label localization in confocal microscopy images.

**`Organelle interaction sites`**: regions of overlap between two or more organelles

These sites can then be measured for features such as number, size, and shape utilizing the `get_morpholgy_metrics()` function outlined in [method_morphology](method_morphology.ipynb) and/or measurements of subcellular distribution utilizing the `get_distribution()` function outlined in [method_distribution](method_distribution.ipynb) notebook.

*You can learn more about how interaction sites are created within infer-subc in the [method_interactions](method_interactions.ipynb) notebook.*

-----

## 🗂️ **Table of Contents**
The following sections are included in this notebook:

**IMPORTS AND LOAD IMAGE**

**EXPLANATION OF STEPS** - This section serves as *expository examples* of the functions used to quantify, batch process, and summarize organelle interactions.

🦠 **Quantify *one or more organelle interaction types* from <ins>ONE IMAGE</ins>**
- Add steps for definining the interaction metric analysis function
- **`STEP 1`** - Create a list of possible interaction types
- **`STEP 2`** - Apply cellmask for single cell analysis
- **`STEP 3`** - Create an overlap for a single interaction
- **`STEP 4`** - Run regionprops for the single interaction
- **`STEP 5`** - Determine organelles involved in interaction by ID
- **`DEFINE`** - The interaction_metric_analysis() function
- **`STEP 6`** - Determine overlaps not present in higher order interactions
- **`DEFINE`** - The find_novel_overlaps() function
- **`STEP 7`** - Loop through the list of possible interactions & quantify them with functions defined above
- **`DEFINE`** - The get_interaction_metrics_3D() function

🧪 **Batch process the quantification of *one or more organelle interaction types* from *multiple images* for a <ins>SINGLE EXPERIMENT</ins>**
- **`STEP 1`** - List images and segmentations to be collected for each image
- **`STEP 2`** - Loop through the list of images and perform the interaction quantification on all interaction types
- **`STEP 3`** - Combine all of the tables together and create/store the csv file
- **`DEFINE`** - The batch_process_interaction() function

🧮 **Summarize organelle interaction metrics *per cell* across <INS>ONE OR MORE EXPERIMENTS</ins>**
- **`STEP 1`** - Get the interaction .csv files
- **`STEP 2`** - Summarize the mean, median, and standard deviation of each feature per cell
- **`STEP 3`** - Calculate additional metrics
- **`STEP 4`** - Unstack the subregions, fill NA values with 0, and save file
- **`DEFINE`** - The batch_interaction_summary_stats() function

**EXECUTE QUANTIFICATION** - Once you understand how the functions work, this section can be used to quantify your data in a quick and easy way.
- **`STEP 1`:** 🧪 **Batch process the quantification of *all possible organelle interaction types* from *multiple cells* for a <ins>SINGLE EXPERIMENT</ins>**
- **`STEP 2`:** 🧮 **Summarize organelle interaction metrics *per cell* across <INS>ONE OR MORE EXPERIMENTS</ins>**

-----
---------------------
## **IMPORTS AND LOAD IMAGE**
Details about the functions included in this subsection are outlined in the [`2.0_quantification_setup`](2.0_quantification_setup.ipynb) notebook. Please visit that notebook first if you are confused about any of the code included here.

#### &#x1F3C3; **Run code; no user input required**

In [1]:
import os
from pathlib import Path
import time
import napari
import numpy as np
import pandas as pd
import itertools
import warnings
from typing import Union
from skimage.measure import label
from infer_subc.core.img import apply_mask
from infer_subc.utils.batch import (list_image_files,
                                    find_segmentation_tiff_files,
                                    make_dict)
from infer_subc.core.file_io import (read_czi_image,
                                     read_tiff_image,
                                     list_image_files)
from infer_subc.quantification.stats import get_morphology_metrics
from infer_subc.quantification.interactions import (all_combo,
                                                    create_overlap,
                                                    interaction_metric_analysis, 
                                                    find_novel_overlaps,
                                                    get_interaction_metrics_3D)
from infer_subc.quantification.distribution import (get_distribution, list_center_objs)

##################
## Color Constants
##################
ORANGE = '#FFA500'
BOPBLUE = '#20ADF8'
MAROON = '#800000'
WHITE = '#FFFFFF'
BLACK = '#000000'

# Splitter Contstant
splitter = 'X'

%load_ext autoreload
%autoreload 2
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

C:\Users\zscoman\AppData\Local\Temp\ipykernel_3804\1642485884.py:6: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: `raw_img_type`, `data_root_path`, `raw_data_path`, `seg_data_path`, and `quant_data_path`.

In [2]:
#### USER INPUT REQUIRED ###
raw_img_type = ".tiff"
data_root_path = Path(os.path.expanduser("~")) / "Documents/Python Scripts/Infer-subc-2D/neurites"
raw_data_path = data_root_path / "raw"
seg_data_path = data_root_path / "segmentations"
quant_data_path = data_root_path / "quant_single"
print(str(quant_data_path))

C:\Users\zscoman\Documents\Python Scripts\Infer-subc-2D\neurites\quant_single


#### &#x1F3C3; **Run code; no user input required**

In [3]:
# Create the output directory to save the segmentation outputs in.
if not Path.exists(quant_data_path):
    Path.mkdir(quant_data_path)
    print(f"making {quant_data_path}")

# Create a list of the file paths for each image in the input folder. Select test image path.
raw_img_file_list = list_image_files(raw_data_path,raw_img_type)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
display(pd.DataFrame({"Image Name":raw_img_file_list}))
pd.set_option('display.max_rows', 5)

,Image Name
0,C:\Users\zscoman\Documents\Python Scripts\Infer-subc-2D\neurites\raw\20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome.tiff


#### &#x1F6D1; &#x270D; **User Input Required:**

Use the list above to specify which image you wish to analyze based on its index: `test_img_n`

In [4]:
#### USER INPUT REQUIRED ###
test_img_n = 0

#### &#x1F3C3; **Run code; no user input required**

In [5]:
# Read in the image and metadata as an ndarray and dictionary from the test image selected above. 
test_img_name = raw_img_file_list[test_img_n]
img_data,meta_dict = read_czi_image(test_img_name)

# Define some of the metadata features.
channel_names = meta_dict['name']
meta = meta_dict['metadata']['aicsimage']
scale = meta_dict['scale']
channel_axis = meta_dict['channel_axis']
file_path = meta_dict['file_name']

print("Metadata information")
print(f"File path: {file_path}")
for i in list(range(len(channel_names))):
    print(f"Channel {i} name: {channel_names[i]}")
print(f"Scale (ZYX): {scale}")
print(f"Channel axis: {channel_axis}")

Metadata information
File path: C:\Users\zscoman\Documents\Python Scripts\Infer-subc-2D\neurites\raw\20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome.tiff
Channel 0 name: 20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome :: Channel:0
Channel 1 name: 20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome :: Channel:1
Channel 2 name: 20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome :: Channel:2
Channel 3 name: 20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome :: Channel:3
Channel 4 name: 20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome :: Channel:4
Channel 5 name: 20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome :: Channel:5
Channel 6 name: 20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome :: Channel:6
Channel 7 name: 20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome :: Channel:7
Scale (ZYX): (0.389118, 0.07064, 0.07064)
Channel axis: 0


#### &#x1F6D1; &#x270D; **User Input Required:**

Specify the following information about the segmentation files: - `org_file_names`, `org_channels_ordered`, `regions_file_names`, `suffix_separator`, and `mask_name`.

In [6]:
#### USER INPUT REQUIRED ###
org_file_names = ["lyso", "mito", "golgi", "perox", "ER", "LD"]
org_channels_ordered = [6, 0, 2, 4, 3, 1]
regions_file_names = ["cell", "nuc", "soma", "neurites"]
subregions_file_names = ["soma", "neurites"]
subregions_multi_instance = [False, False, True]
suffix_separator = "-"
mask_name = "cell"
include_dist = True
dist_centering_obj = ['nuc', 'nuc', None]
dist_center_on=False
dist_keep_center_as_bin=True
dist_zernike_degrees=None
dist_num_bins = 5

#### &#x1F3C3; **Run code; no user input required**

In [7]:
# find file paths for segmentations
all_suffixes = org_file_names + regions_file_names
filez = find_segmentation_tiff_files(file_path, all_suffixes, seg_data_path, suffix_separator)

# read the segmentation and masks/regions files into memory
organelle_segs_list = [read_tiff_image(filez[org]) for org in org_file_names]
organelle_segs = make_dict(list_obj_names = org_file_names, list_obj_segs = organelle_segs_list)

regions = [] 
regions_dict = {}
for m in regions_file_names:
    mfile = read_tiff_image(filez[m])
    regions.append(mfile)
    if m in subregions_file_names:
        regions_dict[m] = mfile
    if m == mask_name:
        mask = mfile

if include_dist:
    centering_objs = list_center_objs(dist_centering_obj, regions, regions_file_names)

# match the intensity channels to the segmentation files
intensities = [img_data[ch] for ch in org_channels_ordered]

# open viewer and add images
viewer = napari.Viewer()
for r, reg in enumerate(regions_file_names):
    viewer.add_image(regions[r],
                     scale=scale,
                     name=f"{reg} mask")

# colors = ["red", "bop orange", "yellow", "green", "blue", "cyan", "magenta", "bop purple"]
for o, org in enumerate(org_file_names):
    viewer.add_image(intensities[o],
                     scale=scale,
                     name=f"{org} intensity channel")
    viewer.add_labels(organelle_segs[org],
                      scale=scale,
                      name=f"{org} segmentation")
viewer.grid.enabled = True
viewer.reset_view()

print("The following matching files were found and can now be viewed in Napari:")
filez

09-Jun-25 14:17:25 - vispy    - WARNING  - QWindowsWindow::setGeometry: Unable to set geometry 1920x1140+0+34 (frame: 1942x1196-11-11) on QWidgetWindow/"_QtMainWindowClassWindow" on "\\.\DISPLAY5". Resulting geometry: 1920x986+0+34 (frame: 1942x1042-11-11) margins: 11, 45, 11, 11 minimum size: 402x570 MINMAXINFO maxSize=0,0 maxpos=0,0 mintrack=826,1196 maxtrack=0,0)


The following matching files were found and can now be viewed in Napari:


{'raw': WindowsPath('C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/raw/20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome.tiff'),
 'lyso': WindowsPath('C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/segmentations/20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome-lyso.tiff'),
 'mito': WindowsPath('C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/segmentations/20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome-mito.tiff'),
 'golgi': WindowsPath('C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/segmentations/20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome-golgi.tiff'),
 'perox': WindowsPath('C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/segmentations/20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome-perox.tiff'),
 'ER': WindowsPath('C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/segmentations/20240118_iN D7 ATAT1 KD_Z 10_Linear unmixing_0_cmle.ome-ER.tiff

------
-----
## **EXPLANATION OF STEPS**

-----
### 🦠 **Quantify *all organelle interaction types* from <ins>ONE IMAGE</ins>**

#### **`STEP 1`** - Apply cell mask for single cell analysis

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** To ensure we are performing single cell analysis, we will apply the cell segmentation as a mask to the segmentation file. This will exclude any objects outside of the mask area from the analysis. The mask file is selected from the list of regions and added to Napari for visual inspection if desired.

In [8]:
# select the mask from the region list
mask = regions[regions_file_names.index(mask_name)]

# add mask to napari for visual inspection
viewer.layers.clear()
viewer.add_image(img_data, scale=scale, name="Intensity Image")
viewer.add_labels(mask, scale=scale, name="Mask")
viewer.grid.enabled = False
viewer.reset_view()



09-Jun-25 14:17:30 - vispy    - WARNING  - QWindowsWindow::setGeometry: Unable to set geometry 1920x1140+0+34 (frame: 1942x1196-11-11) on QWidgetWindow/"_QtMainWindowClassWindow" on "\\.\DISPLAY5". Resulting geometry: 1920x986+0+34 (frame: 1942x1042-11-11) margins: 11, 45, 11, 11 minimum size: 462x570 MINMAXINFO maxSize=0,0 maxpos=0,0 mintrack=946,1196 maxtrack=0,0)


##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The below chunk of code is the step that actually applies the masks to the organelle segmentations. This code has been separated from the above code block to allow users to ensure the correct mask is selected in napari.

In [9]:
# appy mask to segmentations in the organelle segmentation dictionary
for key, org in organelle_segs.items():
    organelle_segs[key] = apply_mask(org, mask)

#### **`STEP 2`** - Create a list of possible interaction types

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This list of possible interactions will be used to both determine which interaction to use as the example interaction for the purposes of this notebook, and will be used in the main function to quantify the interaction metrics of ALL possible interactions 

In [ ]:
all_pos = []
for n in list(map(lambda x:x+2, (range(len(org_file_names)-1)))):
    all_pos += itertools.combinations(org_file_names, n)
possib = [splitter.join(inter) for inter in all_pos]

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.DataFrame({"Overlapping Organelles": possib})
pd.set_option('display.max_rows', 5)

#### **`STEP 3`** - Create a single overlap segmentation

#### &#x1F6D1; &#x270D; **User Input Required:**

Specify the desired overlap to segment. The overlap ID number to choose the desired overlap can be found listed in the output from the previous step.

&#x1F453; **FYI:** This will be used as your example single interaction to examine how the overlaps are quantified, and the rest of the overlaps will be segmented and quantified later in this section of the notebook.

In [11]:
chosen_overlap_ID = 17

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The block of code below performs the `create_overlap()` defined in the `method_interaction.ipynb` notebook. For more information on how our overlaps are made, please see the `method_interaction.ipynb` python notebook for more details.  

In [12]:
interaction_segmentation = create_overlap(orgs=possib[chosen_overlap_ID], 
                                          organelle_segs=organelle_segs, 
                                          splitter=splitter)

#### **`STEP 4`** - Run regionprops for the single chosen interaction



PERSONAL NOTES: MUST ADD NEWEST VERSION OF REGION PROPS & MUST ADD CELL & REGION FINDER TO REGION PROPS

In [13]:
props = get_morphology_metrics(segmentation_img=interaction_segmentation,
                                         seg_name=possib[chosen_overlap_ID],
                                         intensity_img=None,
                                         mask=mask,
                                         mask_name=mask_name,
                                         regions_dict=regions_dict,
                                         scale=scale,
                                         mod=['slice'])

Warning(s) suppressed while quantifying lysoXmitoXER. See 'method_morphology.ipynb' notebook for more details.


#### **`STEP 5`** - Determine organelles involved in interaction by ID

In [14]:
########################################################
## LIST WHICH ORGANELLES ARE INVOLVED IN THE INTERACTION
########################################################
over_inv = []
involved = possib[chosen_overlap_ID].split(splitter)
indexes = {possib[chosen_overlap_ID]: []}
for index, l in enumerate(props["label"]):
        over_inv.clear()
        for org in involved:
            volume = interaction_segmentation[props["slice"][index]]
            lorg = organelle_segs[org][props["slice"][index]]
            volume = volume==l
            lorg = lorg[volume]                                 
            all_inv = np.unique(lorg[lorg>0]).tolist()          
            if len(all_inv) != 1:
                print(f"we have an error.  as-> {all_inv}")
            over_inv.append(f"{all_inv[0]}")
        indexes[possib[chosen_overlap_ID]].append('_'.join(over_inv))

props.rename(columns={'label': 'idx'}, inplace=True)
props.insert(props.columns.get_loc('idx'), 'label',value=indexes[possib[chosen_overlap_ID]])
props.drop(columns=['slice'], inplace=True)
display(props)

,cell_number,subregion,object,label,idx,scale,centroid-0,centroid-1,centroid-2,bbox-0,...,bbox-5,volume,surface_area,SA_to_volume_ratio,equivalent_diameter,extent,euler_number,solidity,axis_major_length,cell_volume
0,cell-1,soma-1,lysoXmitoXER,3_2_1,1,"(0.3891, 0.0706, 0.0706)",0.000000,28.962400,23.169920,0,...,329,0.001942,0.078384,40.368567,0.154785,1.000,1,inf,0.000000,1510.877593
1,cell-1,neurites-1,lysoXmitoXER,7_8_1,2,"(0.3891, 0.0706, 0.0706)",0.172941,40.798524,32.517947,0,...,462,0.017475,0.581696,33.286734,0.321965,0.375,1,0.75,0.900592,1510.877593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224,cell-1,soma-1,lysoXmitoXER,281_12_1,225,"(0.3891, 0.0706, 0.0706)",8.560596,28.538560,23.593760,22,...,335,0.001942,0.156767,80.737133,0.154785,1.000,1,inf,0.000000,1510.877593
225,cell-1,soma-1,lysoXmitoXER,281_12_1,226,"(0.3891, 0.0706, 0.0706)",8.560596,28.679840,23.523120,22,...,334,0.001942,0.156767,80.737133,0.154785,1.000,1,inf,0.000000,1510.877593


#### **`DEFINE`** - interaction_metric_analysis() function

In [15]:
def _interaction_metric_analysis(overlap_ID: str,
                                list_obj_names: list[str],
                                list_obj_segs: list[np.ndarray],
                                mask: np.ndarray,
                                mask_name: str,
                                regions_dict: dict[str:np.ndarray],
                                splitter: str="X",
                                scale: Union[tuple, None]=None,
                                include_dist:bool=False, 
                                dist_centering_obj: Union[np.ndarray, None]=None,
                                dist_num_bins: Union[int, None]=None,
                                dist_zernike_degrees: Union[int, None]=None,
                                dist_center_on: Union[bool, None]=None,
                                dist_keep_center_as_bin: Union[bool, None]=None,
                                return_site: bool=False):
    """
    collect volumentric measurements of intersection between n organelle types

    Parameters
    ------------
    overlap_ID: str
        a value used to describe the organelles present in the overlap that can be divided by the splitter value
    org_dict: dict
        a dictionary of all object segmentations assigned to keys with their objects
    mask: np.ndarray
        3D (ZYX) binary mask of the area to measure interactions from
    splitter: str
        a value used to separate the overlap_ID to determine objects present in overlap
    scale: tuple
        a value present in the metadata determining the scale of the (ZYX) axis
    include_dist:bool=False
        *optional*
        True = include the XY and Z distribution measurements of the overlap sites within the masked region 
        (utilizing the functions get_XY_distribution() and get_Z_distribution() from Infer-subc)
        False = do not include distirbution measurements
    dist_centering_obj: Union[np.ndarray, None]=None
        ONLY NEEDED IF include_dist=True; if None, the center of the mask will be used
        3D (ZYX) np.ndarray containing the object to use for centering the XY distribution mask
    dist_num_bins: Union[int, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is 5
    dist_zernike_degrees: Unions[int, None]=None,
        ONLY NEEDED IF include_dist=True; if None, the zernike share measurements will not be included in the distribution
        the number of zernike degrees to include for the zernike shape descriptors
    dist_center_on: Union[bool, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is False
        True = distribute the bins from the center of the centering object
        False = distribute the bins from the edge of the centering object
    dist_keep_center_as_bin: Union[bool, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is True
        True = include the centering object area when creating the bins
        False = do not include the centering object area when creating the bins


    Regionprops measurements:
    ------------------------
    ['label',
    'centroid',
    'bbox',
    'area',
    'equivalent_diameter',
    'extent',
    'feret_diameter_max',
    'euler_number',
    'convex_area',
    'solidity',
    'axis_major_length',
    'axis_minor_length']

    Additional measurements:
    ----------------------
    ['surface_area']

    
    Returns
    -------------
    pandas dataframe of containing regionprops measurements (columns) for each overlap region (rows)
    
    """
    #########################
    ## CREATE ORG_DICT
    #########################
    org_dict = make_dict(list_obj_names, list_obj_segs)


    #########################
    ## CREATE OVERLAP REGIONS
    #########################
    # run create overlap function
    site = create_overlap(overlap_ID, org_dict, splitter)

    #############################################################################################
    #assert the nth order overlap to within the cellmask
    labels = label(apply_mask(site, mask)).astype(int)

    ###############################
    # RUN REGIONPROPS MEASUREMENTS
    ###############################
     
    props = get_morphology_metrics(segmentation_img=labels,
                                         seg_name=overlap_ID,
                                         intensity_img=None,
                                         mask=mask,
                                         mask_name=mask_name,
                                         regions_dict=regions_dict,
                                         scale=scale,
                                         mod=['slice'])

    ########################################################
    ## LIST WHICH ORGANELLES ARE INVOLVED IN THE INTERACTION
    ########################################################
    over_inv = []
    involved = overlap_ID.split(splitter)
    indexes = {overlap_ID: []}

    for index, l in enumerate(props["label"]):
        over_inv.clear()
        for org in involved:
            volume = labels[props["slice"][index]]
            lorg = org_dict[org][props["slice"][index]]
            volume = volume==l
            lorg = lorg[volume]                                 
            all_inv = np.unique(lorg[lorg>0]).tolist()          
            if len(all_inv) != 1:
                print(f"we have an error.  as-> {all_inv}")
            over_inv.append(f"{all_inv[0]}")
        indexes[overlap_ID].append('_'.join(over_inv))

    props.rename(columns={'label': 'idx'}, inplace=True)
    props.insert(props.columns.get_loc('idx'), 'label',value=indexes[overlap_ID])
    props.drop(columns=['slice'], inplace=True)
    ######################################################
    ## optional: DISTRIBUTION OF INTERACTION MEASUREMENTS
    ######################################################
    if include_dist:
        interaction_dist_tab, XY_bins, XY_wedges = get_distribution(mask=mask,
                                                                    mask_name=mask_name,
                                                                    region_dict=regions_dict,
                                                                    centering_obj=dist_centering_obj,
                                                                    obj=site,
                                                                    obj_name=overlap_ID,
                                                                    scale=scale,
                                                                    num_bins=dist_num_bins,
                                                                    center_on=dist_center_on,
                                                                    keep_center_as_bin=dist_keep_center_as_bin,
                                                                    zernike_degrees=dist_zernike_degrees)
        indexes.clear()
        if return_site:
            return site, props, interaction_dist_tab
        else:
            return props, interaction_dist_tab
    else:
        indexes.clear()
        if return_site:
            return site, props 
        else:
            return props

In [16]:
interaction_table = _interaction_metric_analysis(overlap_ID=possib[chosen_overlap_ID], 
                                                 list_obj_names=org_file_names,
                                                 list_obj_segs=organelle_segs_list,
                                                 mask=mask,
                                                 mask_name=mask_name,
                                                 regions_dict=regions_dict,
                                                 splitter=splitter,
                                                 scale=scale,
                                                 include_dist=False,
                                                 dist_centering_obj=dist_centering_obj,
                                                 dist_num_bins=dist_num_bins,
                                                 dist_zernike_degrees=dist_zernike_degrees,
                                                 dist_center_on=dist_center_on,
                                                 dist_keep_center_as_bin=dist_keep_center_as_bin,
                                                 return_site=False)

print("The interaction quantification here matches the quantification created above:")
print(f"{interaction_table.equals(props)}")

Warning(s) suppressed while quantifying lysoXmitoXER. See 'method_morphology.ipynb' notebook for more details.
The interaction quantification here matches the quantification created above:
True


#### **`STEP 6`** - Segment the overlaps not present in higher order overlaps


In [17]:
LOi_NR = interaction_segmentation.copy()
for org, val in organelle_segs.items():         
    if (org not in possib[chosen_overlap_ID].split(splitter)
        and np.any(interaction_segmentation.astype(int)*val.astype(int))):            
        digit = len(str(np.max(val)))           
        valid = (LOi_NR>0)*(val>0)              
        HOi = (LOi_NR*(10**(digit)))+val        
        HOi[valid.astype(bool)==False]=0        
        HOi = label(HOi)                    
        for num, id in enumerate(np.unique(interaction_segmentation[HOi > 0])):
            LOi_NR[LOi_NR==id] = 0  

#### **`DEFINE`** - find_novel_overlaps() function

In [18]:
def _find_novel_overlaps(site: np.ndarray,
                        orgs: str,
                        organelle_segs: dict[str:np.ndarray],
                        splitter: str="X"):
    ##########################################
    ## DETERMINE NOVEL OVERLAPS
    ##########################################
    LOi_NR = site.copy()                      
    for org, val in organelle_segs.items():         
        if (org not in orgs.split(splitter)
            and np.any(site.astype(int)*val.astype(int))):            
            digit = len(str(np.max(val)))           
            valid = (LOi_NR>0)*(val>0)              
            HOi = (LOi_NR*(10**(digit)))+val        
            HOi[valid.astype(bool)==False]=0        
            HOi = label(HOi)                    
            for num, id in enumerate(np.unique(site[HOi > 0])):
                LOi_NR[LOi_NR==id] = 0    
    return LOi_NR

In [19]:
novel_overlaps = _find_novel_overlaps(site=interaction_segmentation,
                                      orgs=possib[chosen_overlap_ID],
                                      organelle_segs=organelle_segs,
                                      splitter=splitter)

print("The novel overlaps created here match the novel overlaps created above:")
print(f"{(novel_overlaps == LOi_NR).all()}")

The novel overlaps created here match the novel overlaps created above:
True


#### **`STEP 7`** - Loop through each possible interaction to find metrics for each

&#x1F453; **FYI:** In this step, we iterate through all the possible interaction types determined by the prior step, then use both the `find_novel_overlaps()` and `interaction_metric_analysis()` functions, defined in the `method_interaction.ipynb` notebook, to quantify the interaction metrics and determine which interactions are present in higher order overlaps. For more information on how these functions work, please check the `method_interaction.ipynb` python notebook.

In [20]:
inter_tabs = []
if include_dist:
    dist_tabs = []
    for inter in possib:
        print(inter)
        site, inter_tab, dist_tab = _interaction_metric_analysis(overlap_ID=inter,
                                                                list_obj_names=org_file_names,
                                                                list_obj_segs=organelle_segs_list,
                                                                mask=mask,
                                                                mask_name=mask_name,
                                                                regions_dict=regions_dict,
                                                                splitter=splitter,
                                                                scale=scale,
                                                                include_dist=True,
                                                                dist_centering_obj=centering_objs,
                                                                dist_num_bins=dist_num_bins,
                                                                dist_zernike_degrees=dist_zernike_degrees,
                                                                dist_center_on=dist_center_on,
                                                                dist_keep_center_as_bin=dist_keep_center_as_bin,
                                                                return_site=True)
        LOi_NR = _find_novel_overlaps(site, inter, organelle_segs, splitter)
        LOi_NR = apply_mask((LOi_NR>0), mask).astype(int) * site
        redundancy = inter_tab['idx'].isin(np.unique(LOi_NR[LOi_NR>0]).tolist())
        inter_tab.insert((inter_tab.columns.get_loc('label')+1), 
                         "in_higher_order", list(map(bool, ~redundancy)))
        inter_tab.drop(columns=['idx'], inplace=True)
        inter_tabs.append(inter_tab)
        dist_tabs.append(dist_tab)
else:
    for inter in possib:
        site, inter_tab = _interaction_metric_analysis(overlap_ID=inter,
                                                      list_obj_names=org_file_names,
                                                      list_obj_segs=organelle_segs_list,
                                                      mask=mask,
                                                      mask_name=mask_name,
                                                      regions_dict=regions_dict,
                                                      splitter=splitter,
                                                      scale=scale,
                                                      include_dist=False,
                                                      return_site=True)
        LOi_NR = _find_novel_overlaps(site, inter, organelle_segs, splitter)
        LOi_NR = apply_mask((LOi_NR>0), mask).astype(int) * site
        redundancy = inter_tab['idx'].isin(np.unique(LOi_NR[LOi_NR>0]).tolist())
        inter_tab.drop(columns=['idx'], inplace=True)
        inter_tab.insert((inter_tab.columns.get_loc('label')+1), 
                         "in_higher_order", list(map(bool, ~redundancy)))
        inter_tabs.append(inter_tab)

print("Interaction Metrics Table:")
display(pd.concat(inter_tabs.copy(), ignore_index=True))

lysoXmito
Warning(s) suppressed while quantifying lysoXmito. See 'method_morphology.ipynb' notebook for more details.
lysoXgolgi
Warning(s) suppressed while quantifying lysoXgolgi. See 'method_morphology.ipynb' notebook for more details.
lysoXperox
Warning(s) suppressed while quantifying lysoXperox. See 'method_morphology.ipynb' notebook for more details.
lysoXER
Warning(s) suppressed while quantifying lysoXER. See 'method_morphology.ipynb' notebook for more details.
lysoXLD
Warning(s) suppressed while quantifying lysoXLD. See 'method_morphology.ipynb' notebook for more details.
mitoXgolgi
Warning(s) suppressed while quantifying mitoXgolgi. See 'method_morphology.ipynb' notebook for more details.
mitoXperox
Warning(s) suppressed while quantifying mitoXperox. See 'method_morphology.ipynb' notebook for more details.
mitoXER
Warning(s) suppressed while quantifying mitoXER. See 'method_morphology.ipynb' notebook for more details.
mitoXLD
Warning(s) suppressed while quantifying mitoXLD. See

,cell_number,subregion,object,label,in_higher_order,scale,centroid-0,centroid-1,centroid-2,bbox-0,...,bbox-5,volume,surface_area,SA_to_volume_ratio,equivalent_diameter,extent,euler_number,solidity,axis_major_length,cell_volume
0,cell-1,soma-1,lysoXmito,3_2,1.0,"(0.3891, 0.0706, 0.0706)",0.424492,28.750480,23.327255,0,...,333,0.042717,1.164840,27.268471,0.433713,0.244444,1,0.785714,1.543770,1510.877593
1,cell-1,soma-1,lysoXmito,4_3,0.0,"(0.3891, 0.0706, 0.0706)",0.000000,28.962400,24.547400,0,...,349,0.003883,0.134257,34.571947,0.195017,1.000000,1,inf,0.157956,1510.877593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2103,cell-1,soma-1,lysoXgolgiXperoxXER,183_1_54_1,0.0,"(0.3891, 0.0706, 0.0706)",4.280298,26.442907,24.794640,11,...,353,0.005825,0.425281,73.008308,0.223238,0.500000,1,inf,0.290409,1510.877593
2104,cell-1,soma-1,mitoXgolgiXperoxXER,67_1_43_1,0.0,"(0.3891, 0.0706, 0.0706)",4.669416,26.490000,21.615840,12,...,307,0.001942,0.156767,80.737133,0.154785,1.000000,1,inf,0.000000,1510.877593


#### **`DEFINE`** - get_interaction_metrics() function

In [21]:
def _get_interaction_metrics(list_obj_names: list[str],
                             list_obj_segs: list[np.ndarray],
                             mask_name: str,
                             subregion_names: list[str],
                             subregion_multi_instance: list[bool],
                             list_region_segs: Union[list[np.ndarray], None] = None,
                             list_region_names: Union[list[str], None] = None,
                             splitter: str="X",
                             scale: Union[tuple, None]=None,
                             include_dist:bool=False, 
                             dist_centering_obj: Union[np.ndarray, None]=None,
                             dist_num_bins: Union[int, None]=None,
                             dist_zernike_degrees: Union[int, None]=None,
                             dist_center_on: Union[bool, None]=None,
                             dist_keep_center_as_bin: Union[bool, None]=None):
    
    # select the mask from the region list
    mask = list_region_segs[list_region_names.index(mask_name)] #move this to method notebooks

    ########################
    ## CREATE ORGANELLE DICT # add to 2.0
    ########################
    organelle_segs = make_dict(list_obj_names, list_obj_segs)                                                 

    #########################
    ## CREATE SUBREGION DICT # add to 2.0
    #########################
    subregions = {}
    for idx, name in enumerate(subregion_names):
        subregions[name] = list_region_segs[list_region_names.index(name)]
        if((not subregion_multi_instance[idx+1]) 
           and (len(np.unique(subregions[name][subregions[name]!=0])) != 1)):
            raise ValueError(f"Mask {name} is indicated to not be allowed multiple labels according to the masks_multi_instance variable.")
        
    #########################
    ## LIST POSSIBLE OVERLAPS # present in 2.2
    #########################
    possib = all_combo(list_obj_names, splitter)

    ################################
    ## CREATE LIST OF CENTERING OBJS # add to 2.3
    ################################
    if include_dist:
        centering_objs = list_center_objs(dist_centering_obj=dist_centering_obj,
                                          regions=list_region_segs,
                                          regions_names=list_region_names)

    #######################
    ## ANALYZE ALL OVERLAPS
    #######################
    inter_tabs=[]
    dist_tabs=[]
    if include_dist:
        for inter in possib:
            site, inter_tab, dist_tab = _interaction_metric_analysis(overlap_ID=inter,
                                                                    list_obj_names=list_obj_names,
                                                                    list_obj_segs=list_obj_segs,
                                                                    mask=mask,
                                                                    mask_name=mask_name,
                                                                    regions_dict=subregions,
                                                                    splitter=splitter,
                                                                    scale=scale,
                                                                    include_dist=True,
                                                                    dist_centering_obj=centering_objs,
                                                                    dist_num_bins=dist_num_bins,
                                                                    dist_zernike_degrees=dist_zernike_degrees,
                                                                    dist_center_on=dist_center_on,
                                                                    dist_keep_center_as_bin=dist_keep_center_as_bin,
                                                                    return_site=True)
            LOi_NR = _find_novel_overlaps(site, inter, organelle_segs, splitter)
            LOi_NR = apply_mask((LOi_NR>0), mask).astype(int) * site
            novelty = inter_tab['idx'].isin(np.unique(LOi_NR[LOi_NR>0]).tolist())
            inter_tab.insert((inter_tab.columns.get_loc('label')+1), 
                             "in_higher_order", list(map(bool, ~novelty)))
            inter_tab.drop(columns=['idx'], inplace=True)
            inter_tabs.append(inter_tab)
            dist_tabs.append(dist_tab)
        return inter_tabs, dist_tabs
    else:
        for inter in possib:
            site, inter_tab = _interaction_metric_analysis(overlap_ID=inter,
                                                         list_obj_names=list_obj_names,
                                                         list_obj_segs=list_obj_segs,
                                                         mask=mask,
                                                         mask_name=mask_name,
                                                         regions_dict=regions,
                                                         splitter=splitter,
                                                         scale=scale,
                                                         include_dist=False,
                                                         return_site=True)
            LOi_NR = _find_novel_overlaps(site, inter, organelle_segs, splitter)
            LOi_NR = apply_mask((LOi_NR>0), mask).astype(int) * site
            novelty = inter_tab['idx'].isin(np.unique(LOi_NR[LOi_NR>0]).tolist())
            inter_tab.drop(columns=['idx'], inplace=True)
            inter_tab.insert((inter_tab.columns.get_loc('label')+1), 
                             "in_higher_order", list(map(bool, ~novelty)))
            inter_tabs.append(inter_tab)
        return inter_tabs


#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test cell. The settings specified above are applied here.

In [22]:
im_fn = _get_interaction_metrics(list_obj_names=org_file_names,
                                list_obj_segs=organelle_segs_list,
                                mask_name=mask_name,
                                subregion_names=subregions_file_names,
                                subregion_multi_instance=subregions_multi_instance,
                                list_region_segs=regions,
                                list_region_names=regions_file_names,
                                splitter=splitter,
                                scale=scale,
                                include_dist=include_dist,
                                dist_centering_obj=dist_centering_obj,
                                dist_num_bins=dist_num_bins,
                                dist_zernike_degrees=dist_zernike_degrees,
                                dist_center_on=dist_center_on,
                                dist_keep_center_as_bin=dist_keep_center_as_bin)

print("The interaction quantification here matches the quantification created above:")
print(f"{pd.concat(inter_tabs.copy(), ignore_index=True).equals(pd.concat(im_fn[0], ignore_index=True))}")
display(pd.concat(im_fn[0], ignore_index=True))

Warning(s) suppressed while quantifying lysoXmito. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXgolgi. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXperox. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXER. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXLD. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXgolgi. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXperox. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXER. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXLD. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quan

,cell_number,subregion,object,label,in_higher_order,scale,centroid-0,centroid-1,centroid-2,bbox-0,...,bbox-5,volume,surface_area,SA_to_volume_ratio,equivalent_diameter,extent,euler_number,solidity,axis_major_length,cell_volume
0,cell-1,soma-1,lysoXmito,3_2,1.0,"(0.3891, 0.0706, 0.0706)",0.424492,28.750480,23.327255,0,...,333,0.042717,1.164840,27.268471,0.433713,0.244444,1,0.785714,1.543770,1510.877593
1,cell-1,soma-1,lysoXmito,4_3,0.0,"(0.3891, 0.0706, 0.0706)",0.000000,28.962400,24.547400,0,...,349,0.003883,0.134257,34.571947,0.195017,1.000000,1,inf,0.157956,1510.877593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2103,cell-1,soma-1,lysoXgolgiXperoxXER,183_1_54_1,0.0,"(0.3891, 0.0706, 0.0706)",4.280298,26.442907,24.794640,11,...,353,0.005825,0.425281,73.008308,0.223238,0.500000,1,inf,0.290409,1510.877593
2104,cell-1,soma-1,mitoXgolgiXperoxXER,67_1_43_1,0.0,"(0.3891, 0.0706, 0.0706)",4.669416,26.490000,21.615840,12,...,307,0.001942,0.156767,80.737133,0.154785,1.000000,1,inf,0.000000,1510.877593


In [23]:
im_plugin = get_interaction_metrics_3D(list_obj_names=org_file_names,
                                       list_obj_segs=organelle_segs_list,
                                       mask=mask,
                                       mask_name=mask_name,
                                       regions=regions_dict,
                                       splitter=splitter,
                                       scale=scale,
                                       include_dist=include_dist,
                                       dist_centering_obj=centering_objs,
                                       dist_num_bins=dist_num_bins,
                                       dist_zernike_degrees=dist_zernike_degrees,
                                       dist_center_on=dist_center_on,
                                       dist_keep_center_as_bin=dist_keep_center_as_bin)

print("The interaction quantification here matches the quantification created by the plugin:")
print(f"{pd.concat(im_fn[0], ignore_index=True).equals(pd.concat(im_plugin[0], ignore_index=True))}")
display(pd.concat(im_plugin[0], ignore_index=True))

Warning(s) suppressed while quantifying lysoXmito. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXgolgi. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXperox. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXER. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXLD. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXgolgi. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXperox. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXER. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXLD. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quan

,cell_number,subregion,object,label,in_higher_order,scale,centroid-0,centroid-1,centroid-2,bbox-0,...,bbox-5,volume,surface_area,SA_to_volume_ratio,equivalent_diameter,extent,euler_number,solidity,axis_major_length,cell_volume
0,cell-1,soma-1,lysoXmito,3_2,1.0,"(0.3891, 0.0706, 0.0706)",0.424492,28.750480,23.327255,0,...,333,0.042717,1.164840,27.268471,0.433713,0.244444,1,0.785714,1.543770,1510.877593
1,cell-1,soma-1,lysoXmito,4_3,0.0,"(0.3891, 0.0706, 0.0706)",0.000000,28.962400,24.547400,0,...,349,0.003883,0.134257,34.571947,0.195017,1.000000,1,inf,0.157956,1510.877593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2103,cell-1,soma-1,lysoXgolgiXperoxXER,183_1_54_1,0.0,"(0.3891, 0.0706, 0.0706)",4.280298,26.442907,24.794640,11,...,353,0.005825,0.425281,73.008308,0.223238,0.500000,1,inf,0.290409,1510.877593
2104,cell-1,soma-1,mitoXgolgiXperoxXER,67_1_43_1,0.0,"(0.3891, 0.0706, 0.0706)",4.669416,26.490000,21.615840,12,...,307,0.001942,0.156767,80.737133,0.154785,1.000000,1,inf,0.000000,1510.877593


##### &#x1F453; **FYI:** This function has been added to `infer_subc.quantification.interactions` and can be imported with the following:
> ```python
> from infer_subc.quantification.interactions import get_interaction_metrics_3D
> ```

-----
### 🧪 **Batch process interactions across *multiple cells* from <ins>ONE EXPERIMENT</ins>**

#### **`STEP 1` - List images and segmentations to be collected for each**

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** These steps collect a list of the images included in your "raw" (intensity image) data folder. Then, the masks suffixes and organelle suffixes are combined into one list.

In [24]:
# reading list of files from the raw path
img_file_list = list_image_files(raw_data_path, raw_img_type)

# list of organelle segmentation and masks files to collect from each image
segs_to_collect = org_file_names + regions_file_names

#### **`STEP 2` - Loop through the list of images and perform the interaction quantification on all interaction types**

##### &#x1F6D1; &#x270D; **User Input Required:**

Determine if the quantification should be carried out with or without the scale:
- `scale`: True indicates that the function will use the scale metadata to produce "real world" metrics (e.g., microns, etc.). False will produce quantification results in pixel/voxel units.

In [25]:
scale = True

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The block of code below loops through the list of files and runs the `get_interaction_metrics_3D()` function on each one. The loop utilizes the following sequence of steps:
1) Find the paths for all of the organelle and mask segmentation files.
2) Collect the intensity channels and organelle segmentation files in the same order. Store them as lists.
3) Collect all of the region segmentation files in another list.
4) Determine the scale from the metadata.
5) Run the get_interaction_metrics_3D() function and add the resulting data table to the inter_tables list.
6) Repeat the loop above for each image in the raw image list and add them sequentially to the inter_tables list.

In [26]:
# containers to collect data tables
inter_tables = []
dist_tables = []

for img_f in img_file_list:
    all_suffixes = org_file_names + regions_file_names
    filez = find_segmentation_tiff_files(file_path, all_suffixes, seg_data_path, suffix_separator)

    # read in raw file and metadata
    img_data, meta_dict = read_czi_image(filez["raw"])

    # create intensities from raw file as list based on the channel order provided
    intensities = [img_data[ch] for ch in org_channels_ordered]

    # define the scale
    if scale is True:
        scale_tup = meta_dict['scale']
    else:
        scale_tup = None

    # load regions as a list based on order in list (should match order in "masks" file)
    regions = [read_tiff_image(filez[f]) for f in regions_file_names]

    # store organelle images as list
    organelles = [read_tiff_image(filez[org]) for org in org_file_names]
  
    if include_dist:
        inter_tabs, dist_tabs = _get_interaction_metrics(list_obj_names=org_file_names,
                                                        list_obj_segs=organelles,
                                                        mask_name=mask_name,
                                                        subregion_names=subregions_file_names,
                                                        subregion_multi_instance=subregions_multi_instance,
                                                        list_region_segs=regions,
                                                        list_region_names=regions_file_names,
                                                        splitter=splitter,
                                                        scale=scale_tup,
                                                        include_dist=include_dist,
                                                        dist_centering_obj=dist_centering_obj,
                                                        dist_num_bins=dist_num_bins,
                                                        dist_zernike_degrees=dist_zernike_degrees,
                                                        dist_center_on=dist_center_on,
                                                        dist_keep_center_as_bin=dist_keep_center_as_bin)
        for tab in dist_tabs:
            dist_tables.append(tab)
    else:
        inter_tabs = _get_interaction_metrics(list_obj_names=org_file_names,
                                             list_obj_segs=organelles,
                                             mask_name=mask_name,
                                             subregions_file_names=subregions_file_names,
                                             subregions_multi_instance=subregions_multi_instance,
                                             list_region_segs=regions,
                                             list_region_names=regions_file_names,
                                             splitter=splitter,
                                             scale=scale_tup,
                                             include_dist=False)
    for tab in inter_tabs:
        inter_tables.append(tab)

Warning(s) suppressed while quantifying lysoXmito. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXgolgi. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXperox. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXER. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXLD. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXgolgi. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXperox. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXER. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXLD. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quan

#### **`STEP 3` - Combine all of the tables together and create/store the csv file**

##### &#x1F6D1; &#x270D; **User Input Required:**

Select what file name you'd like to use for the output data table:
- `out_file_name`: the prefix you wish to include in the output file name. An underscore will automatically be added to the end of this string before the word "organelles" to indicat this is the results of the organelle morphology analysis.

In [27]:
#### USER INPUT REQUIRED ###
out_file_name = "20241204_test"

##### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block combines the data tables from each image and combines them into one large data table. Then, the file is named using the "*{out_file_name}*_interactions.csv".

In [28]:
# combine all of the image tables together into one sheet
final_inter = pd.concat(inter_tables, ignore_index=True)

# write the new file path include the file name
inter_csv_path = quant_data_path / f"{out_file_name}_interactions.csv"

# save the csv file
final_inter.to_csv(inter_csv_path)
print(f"The following quantification results have been save to {inter_csv_path}.")
final_inter

The following quantification results have been save to C:\Users\zscoman\Documents\Python Scripts\Infer-subc-2D\neurites\quant_single\20241204_test_interactions.csv.


,cell_number,subregion,object,label,in_higher_order,scale,centroid-0,centroid-1,centroid-2,bbox-0,...,bbox-5,volume,surface_area,SA_to_volume_ratio,equivalent_diameter,extent,euler_number,solidity,axis_major_length,cell_volume
0,cell-1,soma-1,lysoXmito,3_2,1.0,"(0.3891, 0.0706, 0.0706)",0.424492,28.750480,23.327255,0,...,333,0.042717,1.164840,27.268471,0.433713,0.244444,1,0.785714,1.543770,1510.877593
1,cell-1,soma-1,lysoXmito,4_3,0.0,"(0.3891, 0.0706, 0.0706)",0.000000,28.962400,24.547400,0,...,349,0.003883,0.134257,34.571947,0.195017,1.000000,1,inf,0.157956,1510.877593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2103,cell-1,soma-1,lysoXgolgiXperoxXER,183_1_54_1,0.0,"(0.3891, 0.0706, 0.0706)",4.280298,26.442907,24.794640,11,...,353,0.005825,0.425281,73.008308,0.223238,0.500000,1,inf,0.290409,1510.877593
2104,cell-1,soma-1,mitoXgolgiXperoxXER,67_1_43_1,0.0,"(0.3891, 0.0706, 0.0706)",4.669416,26.490000,21.615840,12,...,307,0.001942,0.156767,80.737133,0.154785,1.000000,1,inf,0.000000,1510.877593


#### **`DEFINE` - The batch_process_interaction_metrics() function**

In [29]:
def _batch_process_interaction_metrics(raw_path,
                                       seg_path,
                                       out_path,
                                       out_file_name,
                                       raw_file_type,
                                       organelle_names,
                                       region_names,
                                       subregion_names,
                                       subregions_multi_instance,
                                       splitter,
                                       scale,
                                       include_dist,
                                       dist_centering_obj,
                                       dist_num_bins,
                                       dist_zernike_degrees,
                                       dist_center_on,
                                       dist_keep_center_as_bin):
    
    start = time.time()
    count = 0

    # create path objects if inputs are strings
    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(seg_path, str): seg_path = Path(seg_path)
    if isinstance(out_path, str): out_path = Path(out_path)
    
    # create directory is it doesn't exist
    if not Path.exists(out_path):
        Path.mkdir(out_path)
        print(f"Output file path not found. Making {out_path}.")
    

    # reading list of files from the raw path
    img_file_list = list_image_files(raw_path, raw_file_type)
    len_file_list = len(img_file_list)

    # list of organelle segmentation and masks files to collect from each image
    segs_to_collect = organelle_names + region_names

    # containers to collect data tables
    inter_tables = []
    dist_tables = []

    for img_f in img_file_list:
        img_start = time.time()
        count = count + 1
        filez = find_segmentation_tiff_files(file_path, segs_to_collect, seg_data_path, suffix_separator)

        # read in raw file and metadata
        img_data, meta_dict = read_czi_image(filez["raw"])

        # define the scale
        if scale is True:
            scale_tup = meta_dict['scale']
        else:
            scale_tup = None

        # load regions as a list based on order in list (should match order in "masks" file)
        regions = [read_tiff_image(filez[f]) for f in regions_file_names]

        # store organelle images as list
        organelles = [read_tiff_image(filez[org]) for org in org_file_names]
    
        if include_dist:
            inter_tabs, dist_tabs = _get_interaction_metrics(list_obj_names=org_file_names,
                                                            list_obj_segs=organelles,
                                                            mask_name=mask_name,
                                                            subregion_names=subregion_names,
                                                            subregion_multi_instance=subregions_multi_instance,
                                                            list_region_segs=regions,
                                                            list_region_names=regions_file_names,
                                                            splitter=splitter,
                                                            scale=scale_tup,
                                                            include_dist=include_dist,
                                                            dist_centering_obj=dist_centering_obj,
                                                            dist_num_bins=dist_num_bins,
                                                            dist_zernike_degrees=dist_zernike_degrees,
                                                            dist_center_on=dist_center_on,
                                                            dist_keep_center_as_bin=dist_keep_center_as_bin)
            for tab in dist_tabs:
                dist_tables.append(tab)
        else:
            inter_tabs = _get_interaction_metrics(list_obj_names=org_file_names,
                                                 list_obj_segs=organelles,
                                                 mask_name=mask_name,
                                                 subregions_file_names=subregions_file_names,
                                                 subregions_multi_instance=subregions_multi_instance,
                                                 list_region_segs=regions,
                                                 list_region_names=regions_file_names,
                                                 splitter=splitter,
                                                 scale=scale,
                                                 include_dist=False)
        for tab in inter_tabs:
            inter_tables.append(tab)
        end2 = time.time()
        print(f"Completed quantification of {meta_dict['file_name']} in {(end2-img_start)/60} mins.")
        print(f"{count}/{len_file_list} images have been processed.")
        print(f"Time elapsed: {(end2-start)/60} mins")
    final_inter = pd.concat(inter_tables, ignore_index=True)

    inter_csv_path = out_path / f"{out_file_name}_org_morph.csv"
    final_inter.to_csv(inter_csv_path)

    end = time.time()
    print(f"Quantification for {count} files is COMPLETE! Files saved to '{out_path}'.")
    print(f"It took {(end - start)/60} minutes to quantify these files.")

    return final_inter

In [30]:
batch_interaction_table = _batch_process_interaction_metrics(out_file_name=out_file_name,
                                                             raw_path=raw_data_path,
                                                             seg_path=seg_data_path,
                                                             out_path=quant_data_path,
                                                             raw_file_type=raw_img_type,
                                                             organelle_names=org_file_names,
                                                             region_names=regions_file_names,
                                                             subregion_names=subregions_file_names,
                                                             subregions_multi_instance=subregions_multi_instance,
                                                             splitter=splitter,
                                                             scale=scale,
                                                             include_dist=include_dist,
                                                             dist_centering_obj=dist_centering_obj,
                                                             dist_num_bins=dist_num_bins,
                                                             dist_zernike_degrees=dist_zernike_degrees,
                                                             dist_center_on=dist_center_on,
                                                             dist_keep_center_as_bin=dist_keep_center_as_bin)
display(batch_interaction_table)

Warning(s) suppressed while quantifying lysoXmito. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXgolgi. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXperox. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXER. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying lysoXLD. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXgolgi. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXperox. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXER. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quantifying mitoXLD. See 'method_morphology.ipynb' notebook for more details.
Warning(s) suppressed while quan

,cell_number,subregion,object,label,in_higher_order,scale,centroid-0,centroid-1,centroid-2,bbox-0,...,bbox-5,volume,surface_area,SA_to_volume_ratio,equivalent_diameter,extent,euler_number,solidity,axis_major_length,cell_volume
0,cell-1,soma-1,lysoXmito,3_2,1.0,"(0.3891, 0.0706, 0.0706)",0.424492,28.750480,23.327255,0,...,333,0.042717,1.164840,27.268471,0.433713,0.244444,1,0.785714,1.543770,1510.877593
1,cell-1,soma-1,lysoXmito,4_3,0.0,"(0.3891, 0.0706, 0.0706)",0.000000,28.962400,24.547400,0,...,349,0.003883,0.134257,34.571947,0.195017,1.000000,1,inf,0.157956,1510.877593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2103,cell-1,soma-1,lysoXgolgiXperoxXER,183_1_54_1,0.0,"(0.3891, 0.0706, 0.0706)",4.280298,26.442907,24.794640,11,...,353,0.005825,0.425281,73.008308,0.223238,0.500000,1,inf,0.290409,1510.877593
2104,cell-1,soma-1,mitoXgolgiXperoxXER,67_1_43_1,0.0,"(0.3891, 0.0706, 0.0706)",4.669416,26.490000,21.615840,12,...,307,0.001942,0.156767,80.737133,0.154785,1.000000,1,inf,0.000000,1510.877593


-----
### 🧮 **Summarize interactions *per cell* across <INS>ONE OR MORE EXPERIMENTS</ins>**

#### **`STEP 1` - List and collect the interaction csv files**

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: 
- `out_file_name`: The prefix to use when naming the output datatable. Do not add a separator; "_" will be added between your prefix and the base name given in the function below.
- `seg_path`: Path or str to the folder that contains the segmentation tiff files
- `out_path`: Path or str to the folder that the output datatables will be saved to
- `raw_path`: Path or str to the folder that contains the raw image files
- `raw_file_type`: The file type of the raw data; ex - ".tiff", ".czi"
- `organelle_names`: A list of all organelle names that will be analyzed; the names should be the same as the suffix used to name each of the tiff segmentation files. Note: the intensity measurements collect per region (from get_region_morphology_3D function) will only be from channels associated to these organelles 
- `organelle_channels`: A list of channel indices associated to respective organelle staining in the raw image; the indices should listed in same order in which the respective segmentation name is listed in organelle_names
- `region_names`: A list of regions, or masks, to measure; the order should correlate to the order of the channels in the "masks" output segmentation file
- `mask`: The name of the region to use as the mask when measuring the organelles; this should be one of the names listed in regions list; usually this will be the "cell" mask
- `scale`: A tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)
- `seg_suffix`: Any additional text that is included in the segmentation tiff files between the file stem and the segmentation suffix, not including the initial "-"

The defaults below utilize the user input from the `IMPORTS` section.

In [ ]:
out_file_name = "20241204_test"
seg_path = seg_data_path
out_path = quant_data_path
raw_path = raw_data_path
raw_file_type = raw_img_type
organelle_names = org_file_names
organelle_channels = org_channels_ordered
region_names = regions_file_names
mask_name = mask_name
scale = True
seg_suffix = suffix_separator

In [ ]:
# create path list from the inputs given above; if desired, more than one input location can be included here when more than one experimental replicate is included
csv_path_list = [quant_data_path]

ds_count = 0
fl_count = 0
###################
# Read in the csv files and combine them into one of each type
###################
# create empty list to hold the morphology tables from different experiments
inter_tabs = []
inter = "_interaction"

# loop through all of the locations listed above and find the _org_morph files; append them to the list above
for loc in csv_path_list:
    ds_count = ds_count + 1
    files_store = sorted(loc.glob("*.csv"))
    for file in files_store:
        fl_count = fl_count + 1
        stem = file.stem

        if org in stem:
            test_inter = pd.read_csv(file, index_col=0)
            test_inter.insert(0, "dataset", stem[:-len(inter)])
            inter_tabs.append(test_inter)

# combine the org_morph lists found above into one table
inter_df = pd.concat(inter_tabs,axis=0, join='outer')

# print table
inter_df

#### **`STEP 2` - Summarize mean, median, and standard deviation of each feature per cell**

#### **`STEP 3` - Calculate additional metrics**

#### **`STEP 4` - Unstack regions, fill NA values with 0, and save file**

#### **`DEFINE` - batch_interaction_summary_stats() function**